In [1]:
%run ../utility/common_functions.ipynb

In [2]:
run_start_time = datetime.now()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
pipeline_name = "Customer Billing Pipeline"
task_name = "Stage 3 Archive File After Data Load"

In [3]:
# Snowflake stage paths
stage_name = "CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE"

# Get ZIP filename (handles both Azure URL and filename)
zip_file_name = file_name.rsplit("/", 1)[-1]

# Create dated archive destination
archive_folder = datetime.now().strftime("%Y%m%d_%H%M%S")
source_zip_path = f"@{stage_name}/inbound/customer_billing/"
archive_zip_path = f"@{stage_name}/ziparchive/{archive_folder}/"
final_unzip_path = f"@{stage_name}/unzip"

# Log process started
process_log("Start",table_name="",layer="BRONZE",file_name=zip_file_name)

try:
    # Verify original ZIP exists
    zip_files = session.sql(f"LIST {source_zip_path}").collect()

    matching_zip = [
        row for row in zip_files
        if row["name"].rsplit("/", 1)[-1] == zip_file_name
    ]

    if not matching_zip:
        raise FileNotFoundError(f"Original ZIP not found: {zip_file_name}")

    # Copy ZIP to archive folder
    session.sql(f"""COPY FILES INTO {archive_zip_path} FROM {source_zip_path}""").collect()
    print(f"ZIP archived successfully: {archive_zip_path}{zip_file_name}")

    # Delete original ZIP only after successful archival
    session.sql(f"REMOVE {source_zip_path}{zip_file_name}").collect()
    print(f"Original ZIP deleted: {zip_file_name}")

    # Delete extracted folder after successful ZIP archival
    unzip_folder_to_delete = final_unzip_path.rstrip("/")
    print(f"Deleting extracted folder: {unzip_folder_to_delete}")
    session.sql(f"REMOVE {unzip_folder_to_delete}/").collect()
    print(f"Extracted folder deleted successfully: {unzip_folder_to_delete}")

    # Log success
    process_log("Success",table_name="",layer="BRONZE",file_name=zip_file_name)
except Exception as e:
    process_log("Failed",table_name="",layer="BRONZE",file_name=zip_file_name,error_message=str(e))
    raise